In [ ]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--jars /Users/solo/Projects/DataEngineering/.venv/lib/python3.12/site-packages/pyspark/jars/hadoop-aws-3.4.2.jar,"
    "/Users/solo/Projects/DataEngineering/.venv/lib/python3.12/site-packages/pyspark/jars/bundle-2.29.52.jar "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from dotenv import load_dotenv

load_dotenv()

ACCESS_KEY = os.getenv("OCI_ACCESS_KEY")
SECRET_KEY = os.getenv("OCI_SECRET_KEY")
NAMESPACE  = ""
REGION     = ""
ENDPOINT   = f"https://{NAMESPACE}.compat.objectstorage.{REGION}.oraclecloud.com"

spark = SparkSession.builder \
    .appName("OCI-Pipeline") \
    .config("spark.hadoop.fs.s3a.access.key",              ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key",              SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.endpoint",                ENDPOINT) \
    .config("spark.hadoop.fs.s3a.path.style.access",       "true") \
    .config("spark.hadoop.fs.s3a.impl",                    "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.endpoint.region",         REGION) \
    .config("spark.hadoop.fs.s3a.multipart.size",          "104857600") \
    .config("spark.hadoop.fs.s3a.attempts.maximum",        "3") \
    .getOrCreate()

df_raw = spark.read.csv(
    "s3a://de-raw-data/raworder_items_dataset.csv",
    header=True,
    inferSchema=True
)
df_raw.show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/27 01:06:33 WARN Utils: Your hostname, Yashs-Mac-mini.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.3 instead (on interface en1)
26/05/27 01:06:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/05/27 01:06:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [4]:
from pyspark.sql.functions import col, trim, upper, to_timestamp, round as spark_round
from pyspark.sql.functions import year, month

df_clean = df_raw \
    .dropDuplicates() \
    .dropna(subset=["order_id", "product_id"]) \
    .withColumn("price",               col("price").cast("double")) \
    .withColumn("freight_value",       col("freight_value").cast("double")) \
    .withColumn("quantity",            col("order_item_id").cast("integer")) \
    .withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date"))) \
    .withColumn("total_value",         spark_round(col("price") + col("freight_value"), 2)) \
    .filter(col("price") > 0)

print("=== Cleaned schema ===")
df_clean.printSchema()
df_clean.show(5)
print(f"Row count: {df_clean.count()}")

df_partitioned = df_clean \
    .withColumn("year",  year(col("shipping_limit_date"))) \
    .withColumn("month", month(col("shipping_limit_date")))

df_partitioned.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("s3a://de-raw-data/transformed/order_items/")

print("Done — partitioned parquet written to OCI.")


=== Cleaned schema ===
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_value: double (nullable = true)



+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+--------+-----------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|quantity|total_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+--------+-----------+
|0096668e5b0b8e965...|            1|55782cb82e0efe052...|c003204e1ab016dfa...|2018-07-13 13:46:36|166.9|        13.47|       1|     180.37|
|012e93f8170eb75b2...|            1|e3a23a99f074226ad...|efcd8d2104f1a05d0...|2018-02-07 23:51:37| 50.0|         15.1|       1|       65.1|
|012f29911fcd2f120...|            1|8c0bd0d6526379599...|d66c305afaec317eb...|2017-05-22 15:50:13|27.99|        14.08|       1|      42.07|
|013037c3f1773cc8f...|            1|5a6e53c3b4e8684b1...|7299e27ed73d2ad98...|2017-04-03 15:25:21|15.99|        14.52|       1|      30.51|
|027fb2504084c231e..

Row count: 112650


26/05/27 01:21:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/27 01:21:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/27 01:21:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/05/27 01:21:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/05/27 01:21:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/27 01:21:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/27 01:21:33 WARN MemoryManager: Total allocation exceeds 95.00%

Done — partitioned parquet written to OCI.
